# Quick Start

> Minimal Example of StatsForecast

`StatsForecast` follows the sklearn model API. For this minimal example, you will create an instance of the StatsForecast class and then call its `fit` and `predict` methods. We recommend this option if speed is not paramount and you want to explore the fitted values and parameters. 

:::{.callout-tip}
If you want to forecast many series, we recommend using the `forecast` method. Check this [Getting Started with multiple time series](./2_Getting_Started_complete.ipynb) guide. 
:::

The input to StatsForecast is always a data frame in [long format](https://www.theanalysisfactor.com/wide-and-long-data/) with three columns: `unique_id`, `ds` and `y`:

* The `unique_id` (string, int or category) represents an identifier for the series. 

* The `ds` (datestamp) column should be of a format expected by Pandas, ideally YYYY-MM-DD for a date or YYYY-MM-DD HH:MM:SS for a timestamp.

* The `y` (numeric) represents the measurement we wish to forecast. 


As an example, let’s look at the US Air Passengers dataset. This time series consists of monthly totals of a US airline passengers from 1949 to 1960. The CSV is available [here](https://www.kaggle.com/datasets/chirag19/air-passengers).

We assume you have StatsForecast already installed. Check this guide for instructions on [how to install StatsForecast](./0_Installation.ipynb).

First, we’ll  import the data:

In [ ]:
#| hide
# ! pip install statsforecast

In [6]:
import pandas as pd

In [7]:
# write a function to read the data and convert the df to final
def read_data(data_path):
    df = pd.read_csv(data_path)
    final = pd.DataFrame()
    final['ds'] = pd.to_datetime(df['date'])
    final['y'] = df['load_data']
    final['unique_id'] = 'load_data'
    return final
    

In [8]:
path = 'final_output.csv'
final = read_data(path)

final[final['y'].isna()]

,ds,y,unique_id


In [9]:

final.shape


(49632, 3)

In [10]:
final.head()

,ds,y,unique_id
0,2022-04-01 00:15:00,16.07,load_data
1,2022-04-01 00:30:00,13.75,load_data
2,2022-04-01 00:45:00,13.13,load_data
3,2022-04-01 01:00:00,10.36,load_data
4,2022-04-01 01:15:00,9.96,load_data


In [12]:
# plot data using plotly
import plotly.express as px
fig = px.line(final, x="ds", y="y", color='unique_id')
fig.show()

We fit the model by instantiating a new `StatsForecast` object with its two required parameters:
https://nixtla.github.io/statsforecast/src/core/models.html
* `models`: a list of models. Select the models you want from [models](../../src/core/models.ipynb) and import them. For this example, we will use a `AutoARIMA` model. We set `season_length` to 12 because we expect seasonal effects every 12 months. (See: [Seasonal periods](https://robjhyndman.com/hyndsight/seasonal-periods/))

* `freq`: a string indicating the frequency of the data. (See [pandas available frequencies](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases).)

Any settings are passed into the constructor. Then you call its fit method and pass in the historical data frame.

:::{.callout-note}
StatsForecast achieves its blazing speed using JIT compiling through Numba. The first time you call the statsforecast class, the fit method should take around 5 seconds. The second time -once Numba compiled your settings- it should take less than 0.2s. 
:::


In [51]:
%%capture
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

sf = StatsForecast(
    models = [AutoARIMA(season_length = 96)],
    freq = '15T'
)


In [53]:
first_30_days.shape

(2880, 3)

In [55]:
sf.fit(first_30_days)

The `predict` method takes two arguments: forecasts the next `h` (for horizon) and `level`.

* `h` (int): represents the forecast h steps into the future. In this case, 12 months ahead. 

* `level` (list of floats): this optional parameter is used for probabilistic forecasting. Set the `level` (or confidence percentile) of your prediction interval. For example, `level=[90]` means that the model expects the real value to be inside that interval 90% of the times. 

The forecast object here is a new data frame that includes a column with the name of the model and the y hat values, as well as columns for the uncertainty intervals.

In [ ]:
forecast_df = sf.predict(h=96, level=[90]) 

forecast_df.tail()

,ds,AutoARIMA,AutoARIMA-lo-90,AutoARIMA-hi-90
unique_id,,,,
load_data,2023-05-25 23:00:00,14.41084,-35.860405,64.682083
load_data,2023-05-25 23:15:00,14.41084,-36.132881,64.954559
load_data,2023-05-25 23:30:00,14.41084,-36.403896,65.225578
load_data,2023-05-25 23:45:00,14.41084,-36.673470,65.495148
load_data,2023-05-26 00:00:00,14.41084,-36.941631,65.763313


You can plot the forecast by calling the `StatsForecast.plot` method and passing in your forecast dataframe.


In [ ]:

new["ds"]=pd.to_datetime(new["ds"])
sf.plot(new, forecast_df, level=[90], engine='plotly')

:::{.callout-tip}
## Next Steps

* Build and end-to-end forecasting pipeline following best practices in [End to End Walkthrough](./2_Getting_Started_complete.ipynb)
* [Forecast millions of series](../how-to-guides/Prophet_spark_m5.ipynb) in a scalable cluster in the cloud using Spark and Nixtla
* [Detect anomalies](../tutorials/AnomalyDetection.ipynb) in your past observations
:::